# Business Entity Resolution — 100K LoRA Fine-Tuning (SageMaker GPU training job)

Same pipeline as `colab_train_500k.ipynb` (sample → train → full-val eval), but the training runs
as a **SageMaker training job on `ml.g4dn.xlarge` (NVIDIA T4)** instead of on this notebook's CPU:

| Colab 500K notebook | This notebook |
|---|---|
| `%%writefile` the scripts | uses `code/business_entity_resolution/src/` + `sagemaker_job/sm_entry.py` |
| sample on Drive | subsets in S3 (sampled here only if missing) |
| train on the Colab T4 | training job on a T4, ~2 h, ~$1.60 |
| full-val eval in Colab | full-val eval inside the same job |
| results on Drive | results in S3, downloaded to `~/student_resource/models/` |

The job runs on AWS independently of this notebook: you can close the browser, your laptop, or
even stop this space while it trains. Re-run **step 4** any time to check on it.
Hyperparameters match the 500K run: 3 epochs, batch 32, lr 1e-4, seed 42.

## 1. Setup

In [ ]:
import json, os, subprocess, tarfile, time
import boto3

REGION = 'ap-south-1'
BUCKET = 'sagemaker-ap-south-1-845989089398'
PREFIX = 'amazon-entity-resolution'
S3     = f's3://{BUCKET}/{PREFIX}'
ROLE   = 'arn:aws:iam::845989089398:role/service-role/AmazonSageMakerAdminIAMExecutionRole'
IMAGE  = '763104351884.dkr.ecr.ap-south-1.amazonaws.com/pytorch-training:2.8.0-gpu-py312-cu129-ubuntu22.04-sagemaker'
INSTANCE = 'ml.g4dn.xlarge'  # T4; ml.g6.xlarge (L4) is faster but often has no capacity in Mumbai
JOB_PREFIX = 'amazon-ml-cross-100k'

ROOT   = '/home/sagemaker-user/student_resource'
CODE   = f'{ROOT}/code/business_entity_resolution'
DATA   = f'{ROOT}/dataset/train'
MODELS = f'{ROOT}/models'
os.makedirs(MODELS, exist_ok=True)

sm, s3, logs = (boto3.client(s, region_name=REGION) for s in ('sagemaker', 's3', 'logs'))


def s3_exists(key: str) -> bool:
    try:
        s3.head_object(Bucket=BUCKET, Key=key)
        return True
    except s3.exceptions.ClientError:
        return False

## 2. Data in S3: val + 100K subset

Already uploaded (seed-42 splits, identical to the M3 / Colab runs). If either is missing, this
rebuilds it from `train_labeled_pairs.tsv` on this space's disk and uploads it.

In [ ]:
TRAIN_KEY = f'{PREFIX}/dataset/train_100k/train_pairs_100k.tsv'
VAL_KEY   = f'{PREFIX}/dataset/val/val_pairs.tsv'

if not (s3_exists(TRAIN_KEY) and s3_exists(VAL_KEY)):
    if not (os.path.exists(f'{DATA}/val_pairs.tsv') and os.path.exists(f'{DATA}/train_pairs_100k.tsv')):
        !python {CODE}/src/sample_labeled_pairs.py --input {DATA}/train_labeled_pairs.tsv --outdir {DATA}
    !aws s3 cp {DATA}/train_pairs_100k.tsv s3://{BUCKET}/{TRAIN_KEY} --only-show-errors
    !aws s3 cp {DATA}/val_pairs.tsv s3://{BUCKET}/{VAL_KEY} --only-show-errors
!aws s3 ls {S3}/dataset/ --recursive --human-readable

## 3. Launch the training job

Skips launching if a 100K job is already running or has completed — it attaches to that one
instead. Set `FORCE_NEW = True` to start a fresh run. `MaxRuntime` caps cost at ~$7 (5 h).

In [ ]:
FORCE_NEW = False

def latest_job():
    jobs = sm.list_training_jobs(NameContains=JOB_PREFIX, SortBy='CreationTime',
                                 SortOrder='Descending', MaxResults=5)['TrainingJobSummaries']
    for j in jobs:
        if j['TrainingJobStatus'] in ('InProgress', 'Completed'):
            return j['TrainingJobName']
    return None

JOB = None if FORCE_NEW else latest_job()
if JOB:
    print('Attaching to existing job:', JOB)
else:
    # Bundle the unchanged train/eval scripts with the job entry point.
    bundle = '/tmp/sourcedir.tar.gz'
    with tarfile.open(bundle, 'w:gz') as t:
        for f in ('sagemaker_job/sm_entry.py', 'sagemaker_job/requirements.txt',
                  'src/train_cross_encoder.py', 'src/eval_cross_encoder.py'):
            t.add(f'{CODE}/{f}', arcname=os.path.basename(f))
    s3.upload_file(bundle, BUCKET, f'{PREFIX}/jobs/source/sourcedir.tar.gz')

    JOB = f"{JOB_PREFIX}-{time.strftime('%Y%m%d-%H%M%S')}"
    sm.create_training_job(
        TrainingJobName=JOB,
        RoleArn=ROLE,
        AlgorithmSpecification={'TrainingImage': IMAGE, 'TrainingInputMode': 'File'},
        HyperParameters={
            'sagemaker_program': json.dumps('sm_entry.py'),
            'sagemaker_submit_directory': json.dumps(f'{S3}/jobs/source/sourcedir.tar.gz'),
            'sagemaker_region': json.dumps(REGION),
            'epochs': '3', 'batch': '32', 'lr': '0.0001', 'eval-batch': '128', 'tag': json.dumps('100k'),
        },
        InputDataConfig=[
            {'ChannelName': 'train', 'DataSource': {'S3DataSource': {
                'S3DataType': 'S3Prefix', 'S3Uri': f'{S3}/dataset/train_100k/', 'S3DataDistributionType': 'FullyReplicated'}}},
            {'ChannelName': 'val', 'DataSource': {'S3DataSource': {
                'S3DataType': 'S3Prefix', 'S3Uri': f'{S3}/dataset/val/', 'S3DataDistributionType': 'FullyReplicated'}}},
        ],
        OutputDataConfig={'S3OutputPath': f'{S3}/jobs/output/'},
        ResourceConfig={'InstanceType': INSTANCE, 'InstanceCount': 1, 'VolumeSizeInGB': 50},
        StoppingCondition={'MaxRuntimeInSeconds': 5 * 3600},
        Environment={'HF_HUB_DISABLE_TELEMETRY': '1', 'TOKENIZERS_PARALLELISM': 'false'},
    )
    print('Launched:', JOB)
print(f'Console: https://{REGION}.console.aws.amazon.com/sagemaker/home?region={REGION}#/jobs/{JOB}')

## 4. Watch the job

Prints status and the latest log line every minute until the job ends. Interrupting this cell
does **not** stop the job — re-run it any time to check again.

In [ ]:
def last_log_line(job: str) -> str:
    streams = logs.describe_log_streams(logGroupName='/aws/sagemaker/TrainingJobs',
                                        logStreamNamePrefix=job).get('logStreams', [])
    if not streams:
        return '(no logs yet)'
    ev = logs.get_log_events(logGroupName='/aws/sagemaker/TrainingJobs',
                             logStreamName=streams[0]['logStreamName'], limit=20)['events']
    lines = [l for e in ev for l in e['message'].replace('\r', '\n').splitlines() if l.strip()]
    return lines[-1][:200] if lines else '(no logs yet)'

while True:
    d = sm.describe_training_job(TrainingJobName=JOB)
    status, secondary = d['TrainingJobStatus'], d['SecondaryStatus']
    print(time.strftime('%H:%M:%S'), f'{status}/{secondary}', '|', last_log_line(JOB), flush=True)
    if status in ('Completed', 'Failed', 'Stopped'):
        break
    time.sleep(60)

if status != 'Completed':
    print('FailureReason:', d.get('FailureReason'))
else:
    secs = d.get('BillableTimeInSeconds', 0)
    print(f'Billable time: {secs/60:.0f} min')

## 5. Download adapter + metrics

In [ ]:
OUT = f'{MODELS}/cross_encoder_100k'
art = sm.describe_training_job(TrainingJobName=JOB)['ModelArtifacts']['S3ModelArtifacts']
!aws s3 cp {art} /tmp/model.tar.gz --only-show-errors
os.makedirs(OUT, exist_ok=True)
with tarfile.open('/tmp/model.tar.gz') as t:
    t.extractall(OUT)
!ls -lh {OUT}

with open(f'{OUT}/eval_cross_100k.json') as f:
    ev = json.load(f)
print(json.dumps(ev, indent=2)[:3000])

## 6. Summary

The adapter is in `~/student_resource/models/cross_encoder_100k/` and in S3 under
`jobs/output/<job>/output/model.tar.gz`. Compare `eval_cross_100k.json` against `eval_base.json`
(GLiNER baseline) and the 500K run; the metric that matters is
`metrics.source1_macro_f0_5_at_0.500`.

This space isn't needed while the job runs. **Stop it** when you're done so it stops billing.